In [34]:
import os, sys, json, time, platform, datetime as dt
from typing import Literal
import requests, pandas as pd, numpy as np
from pydantic import BaseModel
import openai
from openai import OpenAI

# Create Environment

In [35]:
# ---------- KONFIGURASI (ubah di sini saja) ----------
BASE = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
DECISION_MODEL = "tev1:4b"

# Setiap varian LLM diuji terpisah. reasoning_effort="none" mematikan thinking
# (jika versi Ollama/model Anda menolaknya, hapus varian itu).
LLM_VARIANTS = [
    {
        "name": "llm_default",     
        "model": "qwen3.8:27b", 
        "extra": {}
    },
    {
        "name": "llm_no_thinking", 
        "model": "qwen3.8:27b", 
        "extra": {
            "reasoning_effort": "none"
        }
    },
    # Baseline adil: LLM seukuran decision model, mis.
    # {"name": "llm_small", "model": "<model 4B Anda>", "extra": {"reasoning_effort": "none"}},
]

N_WARMUP   = 3      # request pemanasan, tidak dicatat
N_REPEATS  = 5      # berapa kali seluruh dataset diulang
SEED       = 42
KEEP_ALIVE = "30m"
MAX_TOKENS = 512    # batas output LLM; terlalu kecil -> JSON terpotong
TIMEOUT    = 300
RUN_ID     = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

client = OpenAI(
    base_url=f"{BASE}/v1", 
    api_key="ollama", 
    timeout=TIMEOUT, 
    max_retries=0
)

In [36]:
def _get(path):
    try:
        return requests.get(BASE + path, timeout=10).json()
    except Exception as e:
        return {"error": str(e)}

tags = {
    m["name"]: m.get("digest") for m in _get("/api/tags").get("models", [])
}

wanted = {
    DECISION_MODEL, 
    *[v["model"] for v in LLM_VARIANTS]
}

env = {
    "run_id": RUN_ID,
    "ollama_version": _get("/api/version").get("version"),
    "model_digests": {m: tags.get(m) for m in wanted},
    "python": sys.version.split()[0], "openai_sdk": openai.__version__,
    "platform": platform.platform(),
    "config": {
        "N_WARMUP": N_WARMUP, 
        "N_REPEATS": N_REPEATS, 
        "SEED": SEED,
        "MAX_TOKENS": MAX_TOKENS, 
        "LLM_VARIANTS": LLM_VARIANTS,
        "DECISION_MODEL": DECISION_MODEL
    },
}

missing = [m for m, d in env["model_digests"].items() if d is None]
if missing:
    print("PERINGATAN: model belum di-pull / nama tag berbeda:", missing)

# json.dump(env, open(f"bench_env_{RUN_ID}.json", "w"), indent=2)

env

{'run_id': '20261010_213843',
 'ollama_version': '0.40.2',
 'model_digests': {'tev1:4b': '9b5bb969e46c4b776826d6f2d401e22893205693f172653af6254897255025b8',
  'qwen3.8:27b': '22130167c4c20e20c7b71454612966ca8e8171e9b3cc8ab6ce8aa6cbfec79643'},
 'python': '3.11.3',
 'openai_sdk': '2.45.0',
 'platform': 'Windows-10-10.0.26100-SP0',
 'config': {'N_WARMUP': 3,
  'N_REPEATS': 5,
  'SEED': 42,
  'MAX_TOKENS': 512,
  'LLM_VARIANTS': [{'name': 'llm_default',
    'model': 'qwen3.8:27b',
    'extra': {}},
   {'name': 'llm_no_thinking',
    'model': 'qwen3.8:27b',
    'extra': {'reasoning_effort': 'none'}}],
  'DECISION_MODEL': 'tev1:4b'}}

# Create Dataset

In [37]:
DATA = [
    # (teks, team, refund_diminta)
    ("I was charged twice. Please refund the extra payment.", "billing", True),
    ("The export button throws a 500 error since this morning.", "technical", False),
    ("I forgot my password and the reset email never arrives.", "account", False),
    ("Can you send me last month's invoice as a PDF?", "billing", False),
    ("Your API returns malformed JSON on the /orders endpoint.", "technical", False),
    ("Please delete my account and all my data.", "account", False),
    ("I cancelled yesterday but was still billed. I want my money back.", "billing", True),
    ("The mobile app crashes when I open settings.", "technical", False),
    ("How do I change the email address on my profile?", "account", False),
    ("Do you have an office in Jakarta I can visit?", "other", False),
    ("The product arrived broken, I'd like a refund.", "billing", True),
    ("What are your plans for supporting dark mode?", "other", False),
]
df_data = pd.DataFrame(DATA, columns=["text", "team", "refund"]).rename_axis("item").reset_index()
len(df_data)

12

In [38]:
df_data

,item,text,team,refund
0,0,I was charged twice. Please refund the extra p...,billing,True
1,1,The export button throws a 500 error since thi...,technical,False
2,2,I forgot my password and the reset email never...,account,False
3,3,Can you send me last month's invoice as a PDF?,billing,False
4,4,Your API returns malformed JSON on the /orders...,technical,False
5,5,Please delete my account and all my data.,account,False
6,6,I cancelled yesterday but was still billed. I ...,billing,True
7,7,The mobile app crashes when I open settings.,technical,False
8,8,How do I change the email address on my profile?,account,False
9,9,Do you have an office in Jakarta I can visit?,other,False


# Define Task

In [39]:
TEAM_CRITERIA = {
    "billing":   "Payments, invoices and refunds",
    "technical": "Bugs, errors and integrations",
    "account":   "Login, profile and account management",
    "other":     "None of the above",
}

TEAM_Q   = "Which team should handle this ticket?"
REFUND_Q = "Is the customer requesting a refund?"

class Decision(BaseModel):
    team: Literal["billing", "technical", "account", "other"]
    refund: bool

SYSTEM_PROMPT = (
    "Classify the support ticket.\n"
    f"team: {TEAM_Q} Options: "
    + "; ".join(f"{k} = {v}" for k, v in TEAM_CRITERIA.items())
    + f"\nrefund: {REFUND_Q}"
)

print(SYSTEM_PROMPT)

Classify the support ticket.
team: Which team should handle this ticket? Options: billing = Payments, invoices and refunds; technical = Bugs, errors and integrations; account = Login, profile and account management; other = None of the above
refund: Is the customer requesting a refund?


In [40]:
def call_decision(text):
    body = {
        "model": DECISION_MODEL, 
        "state": {
            "ticket": text
        }, 
        "keep_alive": KEEP_ALIVE,
        "questions": {
            "team":   {
                "type": "choice", 
                "instructions": TEAM_Q, 
                "criteria": TEAM_CRITERIA
            },
            "refund": {
                "type": "noul",   
                "instructions": REFUND_Q
            },
        },
    }

    t0 = time.perf_counter()
    r = requests.post(
        f"{BASE}/v1/systemone", 
        json=body, 
        timeout=TIMEOUT
    )
    lat = time.perf_counter() - t0

    r.raise_for_status()
    j = r.json()
    a = j["answers"]

    return {
        "latency_s": lat, 
        "team": a["team"]["choice"], 
        "refund": a["refund"]["noul"] >= 0.5,
        "confidence": a["team"]["confidence"],
        "input_tokens": j["usage"]["input_tokens"], 
        "output_tokens": j["usage"]["output_tokens"]
    }

In [41]:
_parse = getattr(
    client.chat.completions, 
    "parse", 
    None
) or client.beta.chat.completions.parse

_parse

<bound method Completions.parse of <openai.resources.chat.completions.completions.Completions object at 0x0000017949D57090>>

In [42]:
def call_llm(text, variant):

    t0 = time.perf_counter()
    resp = _parse(
        model=variant["model"],
        messages=[
            {
                "role": "system", 
                "content": SYSTEM_PROMPT
            }, 
            {
                "role": "user", 
                "content": text
            }
        ],
        response_format=Decision, 
        temperature=0, 
        seed=SEED, 
        max_tokens=MAX_TOKENS,
        **variant["extra"],
    )
    lat = time.perf_counter() - t0

    msg = resp.choices[0].message
    if msg.parsed is None:   # refusal / output kosong
        raise ValueError(f"parsed=None finish_reason={resp.choices[0].finish_reason}")
    return {
        "latency_s": lat, 
        "team": msg.parsed.team, 
        "refund": msg.parsed.refund,
        "input_tokens": resp.usage.prompt_tokens, 
        "output_tokens": resp.usage.completion_tokens
    }

# Benchmarking

In [43]:
def unload_all():
    for m in _get("/api/ps").get("models", []):
        try:
            requests.post(
                f"{BASE}/api/generate", 
                json={
                    "model": m["name"], 
                    "keep_alive": 0
                }, 
                timeout=60
            )
        except Exception:
            pass

SYSTEMS = [
    ("decision", lambda t: call_decision(t))
] + [
    (v["name"], (lambda t, v=v: call_llm(t, v))) for v in LLM_VARIANTS
]

print(SYSTEMS)

[('decision', <function <lambda> at 0x0000017949385620>), ('llm_default', <function <listcomp>.<lambda> at 0x0000017948835DA0>), ('llm_no_thinking', <function <listcomp>.<lambda> at 0x0000017948835EE0>)]


In [44]:
# rows, loaded = [], {}
# for name, fn in SYSTEMS:
#     unload_all()
#     try:
#         for i in range(N_WARMUP):
#             fn(df_data.text[i % len(df_data)])
#     except Exception as e:
#         print(f"[{name}] dilewati, warm-up gagal: {type(e).__name__}: {e}")
#         continue
#     loaded[name] = _get("/api/ps")          # bukti model apa yang termuat & porsi GPU/CPU
#     for rep in range(N_REPEATS):
#         for it in df_data.itertuples():
#             row = {"system": name, "rep": rep, "item": it.item, "ok": True, "error": None}
#             try:
#                 row.update(fn(it.text))
#             except Exception as e:
#                 row.update(ok=False, error=f"{type(e).__name__}: {e}"[:200])
#             rows.append(row)
#     print(f"[{name}] selesai")
# unload_all()

# res = pd.DataFrame(rows).merge(df_data[["item", "team", "refund"]], on="item", suffixes=("", "_true"))
# res.to_csv(f"bench_raw_{RUN_ID}.csv", index=False)
# json.dump(loaded, open(f"bench_loaded_{RUN_ID}.json", "w"), indent=2, default=str)
# res.groupby("system").ok.mean()

# Summary

In [45]:
# def summarize(g):
#     ok = g[g.ok]
#     # konsistensi: untuk tiap item, apakah semua pengulangan memberi jawaban yang sama?
#     stable = ok.groupby("item").apply(lambda x: x.team.nunique() == 1 and x.refund.nunique() == 1)
#     return pd.Series({
#         "n": len(g),
#         "valid_rate": g.ok.mean(),                       # objek sesuai schema
#         "acc_team": (ok.team == ok.team_true).mean(),
#         "acc_refund": (ok.refund.astype(bool) == ok.refund_true).mean(),
#         "consistency": stable.mean() if len(stable) else np.nan,
#         "lat_p50_s": ok.latency_s.median(),
#         "lat_p95_s": ok.latency_s.quantile(0.95),
#         "lat_cv": ok.latency_s.std() / ok.latency_s.mean(),
#         "in_tok": ok.input_tokens.mean(),
#         "out_tok": ok.output_tokens.mean(),
#     })

# summary = res.groupby("system").apply(summarize).round(4)
# summary.to_csv(f"bench_summary_{RUN_ID}.csv")
# summary

In [46]:
# # Speedup median terhadap decision model, dengan bootstrap CI 95% (seed tetap)
# rng = np.random.default_rng(SEED)
# def boot_ratio(a, b, n=2000):
#     r = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(b, len(b))) for _ in range(n)]
#     return np.percentile(r, [2.5, 50, 97.5]).round(2)

# okres = res[res.ok]
# base = okres[okres.system == "decision"].latency_s.to_numpy()
# if len(base):
#     for s in okres.system.unique():
#         if s != "decision":
#             lo, mid, hi = boot_ratio(okres[okres.system == s].latency_s.to_numpy(), base)
#             print(f"{s}: {mid}x lebih lambat dari decision model (CI 95%: {lo}x - {hi}x)")
# res[~res.ok].groupby(["system", "error"]).size()

# Pakai Create

In [47]:
import matplotlib
matplotlib.use("Agg")  # simpan grafik ke file, tanpa jendela
import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
import requests
from openai import OpenAI
from pydantic import BaseModel

In [48]:
HISTORY = []

In [ ]:
# ---------------- DEFINISI TUGAS (satu sumber untuk kedua sistem) ----------------
TEAM_CRITERIA = {
    "billing":   "Payments, invoices and refunds",
    "technical": "Bugs, errors and integrations",
    "account":   "Login, profile and account management",
    "other":     "None of the above",
}
TEAM_Q = "Which team should handle this ticket?"
REFUND_Q = "Is the customer requesting a refund?"

class Decision(BaseModel):
    team: Literal["billing", "technical", "account", "other"]
    refund: bool

SYSTEM_PROMPT = (
    "Classify the support ticket.\n"
    f"team: {TEAM_Q} Options: "
    + "; ".join(f"{k} = {v}" for k, v in TEAM_CRITERIA.items())
    + f"\nrefund: {REFUND_Q}"
)

# Sama dengan yang dikirim client.chat.completions.parse(response_format=Decision)
RESPONSE_FORMAT = {
    "type": "json_schema", 
    "json_schema": {
        "name": "Decision", 
        "strict": True,
        "schema": {
            **Decision.model_json_schema(), 
            "additionalProperties": False
        },
    }
}
 
client = OpenAI(base_url=f"{BASE}/v1", api_key="ollama", timeout=TIMEOUT, max_retries=0)
http = requests.Session()  # koneksi dipakai ulang, setara connection pool OpenAI SDK
 
 
def call_decision(text, history=None):
    history = HISTORY if history is None else history
    state = {"conversation": history, "ticket": text} if history else {"ticket": text}
    body = {
        "model": DECISION_MODEL, "state": state, "keep_alive": KEEP_ALIVE,
        "questions": {
            "team":   {"type": "choice", "instructions": TEAM_Q, "criteria": TEAM_CRITERIA},
            "refund": {"type": "noul",   "instructions": REFUND_Q},
        },
    }
    t0 = time.perf_counter()
    r = http.post(f"{BASE}/v1/systemone", json=body, timeout=TIMEOUT)
    lat = time.perf_counter() - t0
    r.raise_for_status()
    j = r.json()
    a = j["answers"]
    return {"latency_s": lat, "team": a["team"]["choice"], "refund": a["refund"]["noul"] >= 0.5,
            "confidence": a["team"]["confidence"],
            "input_tokens": j["usage"]["input_tokens"], "output_tokens": j["usage"]["output_tokens"]}
 
 
def call_llm(text, variant, history=None):
    history = HISTORY if history is None else history
    messages = [
        {
            "role": "system", 
            "content": SYSTEM_PROMPT
        }, 
        *history, 
        {
            "role": "user", 
            "content": text
        }
    ]
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=variant["model"], 
        messages=messages, 
        response_format=RESPONSE_FORMAT,
        temperature=0, 
        seed=SEED, 
        max_tokens=MAX_TOKENS, 
        **variant["extra"],
    )
    lat = time.perf_counter() - t0
    
    choice = resp.choices[0]
    # create() tidak melempar error saat output terpotong/kosong, jadi periksa sendiri
    if choice.finish_reason != "stop":
        raise ValueError(f"finish_reason={choice.finish_reason}")
    d = Decision.model_validate_json(choice.message.content or "")  # ValidationError jika tidak sesuai schema
    return {"latency_s": lat, "team": d.team, "refund": d.refund,
            "input_tokens": resp.usage.prompt_tokens, "output_tokens": resp.usage.completion_tokens}

In [50]:
# ---------------- UTILITAS ----------------
def api_get(path):
    try:
        return requests.get(BASE + path, timeout=10).json()
    except Exception as e:
        return {"error": str(e)}
 
 
def unload_all():
    """Unload semua model agar sistem yang diuji tidak berebut VRAM."""
    for m in api_get("/api/ps").get("models", []):
        try:
            requests.post(f"{BASE}/api/generate", json={"model": m["name"], "keep_alive": 0}, timeout=60)
        except Exception:
            pass
 
 
def record_env(run_id):
    tags = {m["name"]: m.get("digest") for m in api_get("/api/tags").get("models", [])}
    wanted = {DECISION_MODEL, *[v["model"] for v in LLM_VARIANTS]}
    env = {
        "run_id": run_id,
        "ollama_version": api_get("/api/version").get("version"),
        "model_digests": {m: tags.get(m) for m in wanted},
        "python": sys.version.split()[0], "openai_sdk": openai.__version__,
        "platform": platform.platform(),
        "config": {"N_WARMUP": N_WARMUP, "N_REPEATS": N_REPEATS, "SEED": SEED, "MAX_TOKENS": MAX_TOKENS,
                   "LLM_VARIANTS": LLM_VARIANTS, "DECISION_MODEL": DECISION_MODEL, "HISTORY_TURNS": len(HISTORY)},
    }
    missing = [m for m, d in env["model_digests"].items() if d is None]
    if missing:
        print("PERINGATAN: model belum di-pull / nama tag berbeda:", missing)
    with open(f"bench_env_{run_id}.json", "w") as f:
        json.dump(env, f, indent=2)
    print(f"Ollama {env['ollama_version']} | openai SDK {env['openai_sdk']} | run {run_id}")
 
 
def run_benchmark(df_data):
    systems = [("decision", call_decision)] + [
        (v["name"], (lambda t, v=v: call_llm(t, v))) for v in LLM_VARIANTS
    ]
    rows, loaded = [], {}
    for name, fn in systems:
        unload_all()
        try:
            for i in range(N_WARMUP):                 # warm-up: waktu load model tidak ikut terukur
                try:
                    fn(df_data.text[i % len(df_data)])
                except ValueError:                    # output tidak valid saat warm-up: lanjut, nanti tercatat
                    pass
        except Exception as e:                        # server/model tidak bisa dipakai: lewati sistem ini
            print(f"[{name}] dilewati, warm-up gagal: {type(e).__name__}: {e}")
            continue
        loaded[name] = api_get("/api/ps")             # bukti model yang termuat & porsi GPU/CPU
        for rep in range(N_REPEATS):
            for it in df_data.itertuples():
                row = {"system": name, "rep": rep, "item": it.item, "ok": True, "error": None}
                try:
                    row.update(fn(it.text))
                except Exception as e:                # kegagalan dicatat, tidak di-retry
                    row.update(ok=False, error=f"{type(e).__name__}: {e}"[:200])
                rows.append(row)
        print(f"[{name}] selesai")
    unload_all()
    res = pd.DataFrame(rows)
    if res.empty:
        sys.exit("Tidak ada sistem yang berhasil dijalankan. Cek Ollama dan nama model.")
    res = res.merge(df_data[["item", "team", "refund"]], on="item", suffixes=("", "_true"))
    return res, loaded
 
 
def summarize(g):
    ok = g[g.ok]
    # konsistensi: untuk tiap item, apakah semua pengulangan memberi jawaban yang sama?
    stable = ok.groupby("item")[["team", "refund"]].nunique().eq(1).all(axis=1)
    return pd.Series({
        "n": len(g),
        "valid_rate": g.ok.mean(),                    # objek sesuai schema
        "acc_team": (ok.team == ok.team_true).mean(),
        "acc_refund": (ok.refund.astype(bool) == ok.refund_true).mean(),
        "consistency": stable.mean() if len(stable) else np.nan,
        "lat_p50_s": ok.latency_s.median(),
        "lat_p95_s": ok.latency_s.quantile(0.95),
        "lat_cv": ok.latency_s.std() / ok.latency_s.mean(),
        "in_tok": ok.input_tokens.mean(),
        "out_tok": ok.output_tokens.mean(),
    })
 
 
def print_speedup(res):
    """Speedup median terhadap decision model, bootstrap CI 95% dengan seed tetap."""
    rng = np.random.default_rng(SEED)
    ok = res[res.ok]
    base = ok[ok.system == "decision"].latency_s.to_numpy()
    if not len(base):
        return
    for s in ok.system.unique():
        if s == "decision":
            continue
        a = ok[ok.system == s].latency_s.to_numpy()
        r = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(base, len(base))) for _ in range(2000)]
        lo, mid, hi = np.percentile(r, [2.5, 50, 97.5]).round(2)
        print(f"{s}: {mid}x lebih lambat dari decision model (CI 95%: {lo}x - {hi}x)")
 
 
def main():
    run_id = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
    record_env(run_id)
    df_data = pd.DataFrame(DATA, columns=["text", "team", "refund"]).rename_axis("item").reset_index()
 
    res, loaded = run_benchmark(df_data)
    res.to_csv(f"bench_raw_{run_id}.csv", index=False)
    with open(f"bench_loaded_{run_id}.json", "w") as f:
        json.dump(loaded, f, indent=2, default=str)
 
    summary = pd.DataFrame({name: summarize(g) for name, g in res.groupby("system")}).round(4)
    summary.to_csv(f"bench_summary_{run_id}.csv")
    print("\n=== RINGKASAN ===")
    print(summary.to_string())
    print()
    print_speedup(res)
 
    errors = res[~res.ok].groupby(["system", "error"]).size()
    if len(errors):
        print("\n=== KEGAGALAN ===")
        print(errors.to_string())
 
    ax = res[res.ok].boxplot(column="latency_s", by="system", figsize=(7, 4))
    ax.set_yscale("log")
    ax.set_ylabel("latency per request (detik, skala log)")
    ax.set_title("")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig(f"bench_latency_{run_id}.png", dpi=120)
    print(f"\nFile hasil tersimpan dengan akhiran _{run_id}")
 
 
if __name__ == "__main__":
    main()


Ollama 0.40.2 | openai SDK 2.45.0 | run 20261010_213844
[decision] selesai
[llm_default] selesai
[llm_no_thinking] selesai

=== RINGKASAN ===
             decision  llm_default  llm_no_thinking
n             60.0000      60.0000          60.0000
valid_rate     1.0000       1.0000           1.0000
acc_team       1.0000       1.0000           1.0000
acc_refund     1.0000       1.0000           1.0000
consistency    1.0000       1.0000           1.0000
lat_p50_s      0.3866      10.7280           2.5977
lat_p95_s      0.4811      16.3209           4.1928
lat_cv         0.1002       0.2534           0.2339
in_tok       492.8333      86.0833          88.0833
out_tok        3.0000     116.0000          16.5000

llm_default: 27.79x lebih lambat dari decision model (CI 95%: 25.44x - 31.46x)
llm_no_thinking: 6.71x lebih lambat dari decision model (CI 95%: 6.35x - 7.08x)

File hasil tersimpan dengan akhiran _20261010_213844
